In [0]:
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.client import MlflowClient
from xgboost import XGBClassifier
import json, xgboost as xgb
from datetime import datetime
import pickle
import numpy as np
import pandas as pd
from typing import Iterable, Sequence, Tuple, Optional

In [0]:
%run ../../config/utils

In [0]:
def random_numeric_df(
    columns: Sequence[str],
    rows: int = 5,
    seed: Optional[int] = None,
    float_range: Tuple[float, float] = (0.0, 1.0),
    as_integers: bool = False,
    int_range: Tuple[int, int] = (0, 100)
) -> pd.DataFrame:
    """
    Create a DataFrame with `rows` rows of random numeric values for the given `columns`.

    - Set `seed` for reproducibility.
    - By default returns floats in `float_range`.
    - Set `as_integers=True` to return ints in `int_range` (inclusive of low, exclusive of high).
    """
    if not isinstance(columns, Iterable) or isinstance(columns, (str, bytes)):
        raise ValueError("`columns` must be a sequence of column names (e.g., list of strings).")

    rng = np.random.default_rng(seed)
    n_cols = len(columns)

    if as_integers:
        low, high = int_range
        data = rng.integers(low, high, size=(rows, n_cols), dtype=np.int64)
    else:
        low, high = float_range
        data = rng.uniform(low, high, size=(rows, n_cols))

    return pd.DataFrame(data, columns=list(columns))

In [0]:
features_dataset = spark.read.parquet('s3://memberanalytics-data-out-prod/MODELDATA/Digital_propensity_scoring/Features/Digital_Propensity_Features_2025-11-01')

In [0]:
experiment_name = '/Workspace/Shared/pe_memberdna/digital_propensity_model'

mlflow.xgboost.autolog(disable=False, log_input_examples=True, log_models=False, log_datasets=False)
mlflow.sklearn.autolog(disable=False, log_input_examples=True, log_models=False, log_datasets=False)

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri('databricks-uc')

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name) 
mlflow.set_experiment(experiment_name)

In [0]:
mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')

with mlflow.start_run(run_name=f'importing_trips_model_{mark_datetime}') as run:
    clf = xgb.XGBClassifier()
    clf.load_model(f"/Volumes/{catalog_name}/pe/helpers/digital_propensity/trips_model.json")

    with open(f'/Volumes/{catalog_name}/pe/helpers/digital_propensity/trips_model.meta.json', 'r') as f:
        metadata = json.load(f)

    mlflow.log_params(metadata)
    mlflow.log_params(metadata['xgb_params'])


    test_df = features_dataset.select(*metadata['feature_names']).limit(5).toPandas()

    example_in = test_df
    example_out = clf.predict_proba(example_in)[:, 1]
    signature = infer_signature(test_df, example_out)

    model_info = mlflow.xgboost.log_model(
        xgb_model=clf,
        artifact_path="model",
        signature=signature,
        input_example=example_in,
        registered_model_name=f"{catalog_name}.pe.digital_trips_model"  # UC path if desired
    )

In [0]:
client = MlflowClient()
client.set_registered_model_alias(f"{catalog_name}.pe.digital_trips_model", 'champion', model_info.registered_model_version)

In [0]:
mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')

with mlflow.start_run(run_name=f'importing_sales_model_{mark_datetime}') as run:
    clf = xgb.XGBClassifier()
    clf.load_model(f"/Volumes/{catalog_name}/pe/helpers/digital_propensity/sales_model.json")

    with open(f'/Volumes/{catalog_name}/pe/helpers/digital_propensity/sales_model.meta.json', 'r') as f:
        metadata = json.load(f)

    mlflow.log_params(metadata)
    mlflow.log_params(metadata['xgb_params'])


    test_df = features_dataset.select(*metadata['feature_names']).limit(5).toPandas()

    example_in = test_df
    example_out = clf.predict_proba(example_in)[:, 1]
    signature = infer_signature(test_df, example_out)

    model_info = mlflow.xgboost.log_model(
        xgb_model=clf,
        artifact_path="model",
        signature=signature,
        input_example=example_in,
        registered_model_name=f"{catalog_name}.pe.digital_sales_model"  
    )

In [0]:
client = MlflowClient()
client.set_registered_model_alias(f"{catalog_name}.pe.digital_sales_model", 'champion', model_info.registered_model_version)